# Task 1 — Prove Why We Visualize

**Dataset:** `datasaurus.csv` — 1,846 rows, 3 columns: `dataset`, `x`, `y`  
13 different datasets of 142 points each.

**Goal:** Demonstrate that identical summary statistics can hide radically different data shapes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Load data
df = pd.read_csv('../data/datasaurus.csv') if False else pd.read_csv('data/datasaurus.csv')
print(f'Shape: {df.shape}')
print(f'Datasets: {sorted(df["dataset"].unique())}')
print(df.head())

## Step 1 — Summary Statistics for All 13 Datasets

In [ ]:
stats = df.groupby('dataset').apply(
    lambda g: pd.Series({
        'mean_x':    round(g['x'].mean(), 2),
        'mean_y':    round(g['y'].mean(), 2),
        'std_x':     round(g['x'].std(),  2),
        'std_y':     round(g['y'].std(),  2),
        'corr_xy':   round(g['x'].corr(g['y']), 2),
        'n':         len(g)
    })
).reset_index()

print(stats.to_string(index=False))

## Step 2 — What Would I Conclude from the Table Alone?

Looking at the summary table, every dataset has:
- Mean of x ≈ 54.3, mean of y ≈ 47.8
- Standard deviation of x ≈ 16.8, standard deviation of y ≈ 26.9
- Correlation ≈ −0.06

**Conclusion from table alone:** If this table were all I had, I would conclude that all 13 datasets are essentially identical — they share the same center, spread, and (near-zero) linear relationship. I would assume they were 13 random samples from the same underlying population and that any one of them is as representative as any other. There would be no reason to distinguish them.

## Step 3 — Grid of Scatter Plots for All 13 Datasets

In [ ]:
datasets = sorted(df['dataset'].unique())
n_cols = 4
n_rows = (len(datasets) + n_cols - 1) // n_cols  # = 4 rows

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 12))
fig.suptitle(
    'Identical Summary Statistics Hide 13 Completely Different Shapes',
    fontsize=14, fontweight='bold', y=1.01
)

axes_flat = axes.flatten()

for i, name in enumerate(datasets):
    ax = axes_flat[i]
    subset = df[df['dataset'] == name]
    ax.scatter(subset['x'], subset['y'], s=10, alpha=0.7, color='steelblue', linewidths=0)
    ax.set_title(name, fontsize=10, fontweight='bold')
    ax.set_xlabel('x', fontsize=8)
    ax.set_ylabel('y', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)

# Hide unused axes
for j in range(len(datasets), len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.tight_layout()
plt.savefig('task1_datasaurus_grid.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as task1_datasaurus_grid.png')

**Why scatter plots?**  
A scatter plot is the only chart that reveals the joint distribution of two continuous variables simultaneously. Each point represents one observation's (x, y) pair, making it possible to see shapes, clusters, and outliers that summary statistics entirely suppress. For this task it is the only honest choice — any aggregated chart (bar, line, histogram) would itself summarize the data and miss the point.

## Step 4 — What I Actually Found

The 13 datasets are shockingly different. The first one — `dino` — draws a dinosaur. Others include a star (`star`), a bullseye (`bullseye`), two distinct point clouds (`wide_lines`, `dots`), and a classic X shape (`x_shape`). Several form geometric shapes: circles, slanted lines, high-density blobs. Despite this dramatic visual variation, every dataset yields a mean, standard deviation, and correlation coefficient that rounds to the same value to two decimal places.

**Answer to the key question:** Having 13 datasets with 142 points each (versus Anscombe's 4 with 11 points) does **not** make summary statistics more trustworthy — it makes the demonstration *more* alarming. With only 4 datasets and 11 points, one might argue the sample is too small for statistics to stabilize. The Datasaurus shows that even with 142 points — a perfectly respectable sample size — the mean, standard deviation, and correlation can be identical across wildly different shapes. The problem is not sample size; it is that these statistics are **model-dependent**. They measure specific features of a distribution (its center, spread, and linear co-movement) and are entirely blind to shape. No amount of additional data will make a non-linear or non-elliptical structure visible in a table of means and correlations. Only a plot can show it.